# Статистический анализ точности системы лазерного наведения с БПЛА

Анализ посвящён исследованию точности системы лазерного наведения, установленной на квадрокоптере со стабилизированным подвесом.

Основная метрика качества — **радиальная ошибка** $R = \sqrt{r_x^2 + r_y^2}$ (мм), измеренная как горизонтальное отклонение лазерного пятна от реперной точки на поверхности.

### Характеристика выборки

| Параметр | Значение |
|---|---|
| Период проведения испытаний | апрель — октябрь 2025 г. |
| Число лётных дней | 60 |
| Контрольные точки | 5 (P1–P5) |
| Высоты зависания | $H \in \{3,\; 5,\; 10,\; 15,\; 20\}$ м |
| Повторов на каждую комбинацию (точка × высота) | 3 |
| Общее число измерений | $60 \times 5 \times 5 \times 3 = 4500$ |

Для каждого измерения фиксировались: время, координаты точки, высота $H$, номер повтора, локальная скорость ветра $V$ (м/с), порывы ветра, направление ветра, облачность (%), число спутников GNSS, а также компоненты отклонения $r_x$, $r_y$ и радиальная ошибка $R$.

In [25]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
JSON_PATH = BASE_DIR / "balanced_selection" / "selected_days_2024-2025.json"

H_ORDER = [3, 5, 10, 15, 20]
H_CAT_ORDER = ["3 м", "5 м", "10 м", "15 м", "20 м"]
ALPHA = 0.05

COL_MAP = {
    "Время": "time", "Точка": "point", "H (м)": "H", "Полёт": "flight",
    "V (м/с)": "V", "Порывы, откр. источник (м/с)": "gusts",
    "Направление ветра": "wind_dir", "Облачность (%)": "cloud",
    "Спутники": "sat", "r_x (мм)": "r_x", "r_y (мм)": "r_y", "R (мм)": "R"
}


def load_measurements(input_dir: Path) -> pd.DataFrame:
    frames = []
    for fpath in sorted(input_dir.glob("*.xlsx")):
        tmp = pd.read_excel(fpath, header=12, engine="openpyxl")
        tmp.rename(columns=COL_MAP, inplace=True)
        tmp["date"] = fpath.stem
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    for c in ("H", "V", "cloud", "sat", "R", "r_x", "r_y", "gusts", "wind_dir"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.dropna(subset=["H", "V", "R"], inplace=True)
    return df


weather = pd.DataFrame(json.loads(JSON_PATH.read_text(encoding="utf-8")))
weather["date"] = pd.to_datetime(weather["date"])

df = load_measurements(OUTPUT_DIR)
df["H_cat"] = df["H"].astype(int).astype(str) + " м"
df["V_bin"] = pd.qcut(
    df["V"], 5,
    labels=["очень тихо", "тихо", "умеренно", "свежо", "сильно"],
    duplicates="drop"
)

print(f"Загружено: {len(df)} измерений за {df['date'].nunique()} дней")
print(f"Погодный JSON: {len(weather)} дней")
print(f"Высоты: {sorted(df['H'].unique())} м")
print(f"Точки: {', '.join(sorted(df['point'].unique()))}")
print(f"Повторов на высоту: {df['flight'].nunique()}")

Загружено: 4500 измерений за 60 дней
Погодный JSON: 60 дней
Высоты: [np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20)] м
Точки: P1, P2, P3, P4, P5
Повторов на высоту: 3


### 1. Обзор выборки и условий эксперимента

Перед анализом ошибки наведения необходимо убедиться, что выборка лётных дней охватывает достаточно разнообразные метеорологические условия и не содержит систематического смещения по какому-либо фактору. Ниже приведено распределение дней по типу погоды и гистограммы ключевых метеопараметров.

In [26]:
weather_counts = weather["weather_code_text"].value_counts().reset_index()
weather_counts.columns = ["Погода", "Дней"]

fig = px.bar(
    weather_counts, x="Погода", y="Дней",
    color="Погода", text="Дней",
    title="Рис. 1. Распределение лётных дней по типу погоды",
)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Количество дней")
fig.update_traces(textposition="outside")
fig.show()

In [27]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Температура (°C)", "Средний ветер (м/с)",
    "Максимальные порывы (м/с)", "Облачность (%)"
])

for i, (col, name) in enumerate([
    ("temperature_2m_mean", "Температура"),
    ("wind_speed_10m_mean", "Ветер"),
    ("wind_gusts_10m_max", "Порывы"),
    ("cloud_cover_mean", "Облачность"),
]):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=weather[col], nbinsx=15, name=name, showlegend=False),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=500, title_text="Рис. 2. Распределение метеопараметров по дням выборки")
fig.show()

In [28]:
fig = px.histogram(
    df, x="R", nbins=80, marginal="box",
    title="Рис. 3. Общее распределение радиальной ошибки R",
    labels={"R": "R (мм)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(yaxis_title="Количество измерений", height=400)
fig.show()

Выборка охватывает разнообразные погодные условия — от ясных дней со слабым ветром до пасмурных с порывами. Ветровые режимы распределены равномерно благодаря сбалансированной выборке дней.

### 2. Описательная статистика

Для первичной характеристики выборки вычислим основные числовые показатели радиальной ошибки $R$ в каждой высотной группе: объём подвыборки $N$, среднее $\bar{R}$, стандартное отклонение $\sigma$, минимум, максимум и 95-й процентиль $R$

In [46]:
desc = (
    df.groupby("H", sort=True)["R"]
    .agg(
        N="count",
        Среднее="mean",
        Ст_откл="std",
        Минимум="min",
        Максимум="max",
        P95=lambda x: x.quantile(0.95)
    )
    .round(1)
)
desc.index.name = "H (м)"
desc.columns = ["N", "Среднее, мм", "σ, мм", "Мин, мм", "Макс, мм", "R₉₅, мм"]

fig = go.Figure(data=[go.Table(
    columnwidth=[50, 50, 80, 70, 60, 70, 70],
    header=dict(
        values=["<b>H (м)</b>"] + [f"<b>{c}</b>" for c in desc.columns],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=[desc.index.tolist()] + [desc[c].tolist() for c in desc.columns],
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 1. Описательная статистика радиальной ошибки R по высотам",
    width=820, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
fig.show()

n_per_h = len(df) // df["H"].nunique()
n_days = df["date"].nunique()
print(f"{n_per_h} измерений на высоту: {n_days} дней × 5 точек × 3 повтора = {n_per_h}")

print("\nОписательная статистика непрерывных ковариат:")
for col, unit in [("V", "м/с"), ("gusts", "м/с"), ("cloud", "%"), ("sat", "шт.")]:
    vals = df[col].dropna()
    print(f"  {col}: среднее = {vals.mean():.1f} {unit}, σ = {vals.std():.1f}, "
          f"диапазон [{vals.min():.1f} – {vals.max():.1f}]")

900 измерений на высоту: 60 дней × 5 точек × 3 повтора = 900

Описательная статистика непрерывных ковариат:
  V: среднее = 3.4 м/с, σ = 1.4, диапазон [0.8 – 8.8]
  gusts: среднее = 9.9 м/с, σ = 3.1, диапазон [3.2 – 19.0]
  cloud: среднее = 59.9 %, σ = 33.7, диапазон [0.0 – 100.0]
  sat: среднее = 21.1 шт., σ = 3.6, диапазон [16.0 – 31.0]


По описательным характеристикам уже на раннем этапе видны две важные закономерности:

1. **Монотонный рост среднего** $\bar{R}(H)$ с увеличением высоты, что свидетельствует о систематическом влиянии геометрического рычага: при наклоне подвеса на угол $\delta\varphi$ горизонтальное смещение составляет $\Delta R \approx H \cdot \sin(\delta\varphi)$, т.е. линейно возрастает с высотой.

2. **Стандартное отклонение растёт вместе со средним** — чем выше $H$, тем больше не только среднее $\bar{R}$, но и разброс $\sigma$. Это означает, что ошибка ведёт себя пропорционально: на большой высоте и разброс крупнее, и их непредсказуемость выше.

Обе закономерности являются первыми косвенными свидетельствами доминирующего влияния фактора высоты $H$ и будут подтверждены формально в разделах дисперсионного анализа.


### 3. Распределение ошибки по высотам

In [30]:
fig = px.violin(
    df, x="H", y="R", color="H_cat", box=True, points=False,
    title="Рис. 4. Распределение радиальной ошибки R по высотам",
    labels={"H": "Высота H (м)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_layout(showlegend=False, height=600, width=900)
fig.show()

In [31]:
stats_by_h = df.groupby("H")["R"].agg(["mean", "std", "count"]).reset_index()
stats_by_h["se"] = stats_by_h["std"] / np.sqrt(stats_by_h["count"])
stats_by_h["ci95"] = stats_by_h["se"] * 1.96

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=stats_by_h["H"], y=stats_by_h["mean"],
    mode="lines+markers", name="Среднее R",
    line=dict(width=3), marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=list(stats_by_h["H"]) + list(stats_by_h["H"][::-1]),
    y=list(stats_by_h["mean"] + stats_by_h["ci95"])
      + list((stats_by_h["mean"] - stats_by_h["ci95"])[::-1]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)",
    line=dict(width=0), name="95% дов. интервал", showlegend=True,
))
fig.update_layout(
    title="Рис. 5. Средняя ошибка R по высоте (с 95% доверительным интервалом)",
    xaxis_title="Высота H (м)", yaxis_title="R (мм)", height=400,
)
fig.show()

На рис. 4 отчётливо видно, что с ростом высоты $H$:

- **медиана** и **среднее** $R$ систематически смещаются вверх;
- **межквартильный размах** расширяется — разброс ошибок увеличивается;
- **правый хвост** распределения удлиняется, что указывает на появление больших выбросов при неблагоприятных условиях.

Физическая интерпретация: при угловой нестабильности подвеса $\delta\varphi$ горизонтальное отклонение составляет

$$\Delta R \approx H \cdot \tan(\delta\varphi) \approx H \cdot \delta\varphi,$$

т.е. ошибка линейно пропорциональна высоте при малых углах. На рис. 5 подтверждается монотонный рост $\bar{R}(H)$ с неперекрывающимися доверительными интервалами на всех высотах.

### 4. Эффект плато на высотах 15–20 м

Испытания проводились на территории карьера. На высотах до 10–15 м квадрокоптер находится внутри карьерной выемки, где рельеф гасит ветровое воздействие. При подъёме на 15–20 м аппарат вылетает над бровкой карьера и попадает в открытый ветровой поток.

Ожидаемые эффекты:
- рост ошибки $R$ с высотой **замедляется** (эффект плато), поскольку аэродинамическое демпфирование стабилизируется;
- **чувствительность к ветру возрастает** — разница между тихим и ветреным днём увеличивается.

In [32]:
v_q25 = df["V"].quantile(0.25)
v_q75 = df["V"].quantile(0.75)

low_wind = df[df["V"] <= v_q25].groupby("H")["R"].median().reset_index()
low_wind.columns = ["H", "R_median"]
low_wind["Режим"] = f"Слабый ветер (V ≤ {v_q25:.1f} м/с)"

high_wind = df[df["V"] >= v_q75].groupby("H")["R"].median().reset_index()
high_wind.columns = ["H", "R_median"]
high_wind["Режим"] = f"Сильный ветер (V ≥ {v_q75:.1f} м/с)"

plateau_df = pd.concat([low_wind, high_wind])

fig = px.line(
    plateau_df, x="H", y="R_median", color="Режим",
    markers=True,
    title="Рис. 6. Медианная ошибка R по высоте: слабый и сильный ветер",
    labels={"H": "Высота H (м)", "R_median": "Медианная R (мм)"},
)
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(height=450)
fig.show()

In [48]:
heights = sorted(df["H"].unique())
median_by_h = df.groupby("H")["R"].median()

deltas = []
for i in range(1, len(heights)):
    h_prev, h_curr = heights[i - 1], heights[i]
    delta = median_by_h[h_curr] - median_by_h[h_prev]
    deltas.append({"Медианная ошибка по высотам": f"{int(h_prev)}→{int(h_curr)} м", "ΔR": delta})

delta_df = pd.DataFrame(deltas)

fig = px.bar(
    delta_df, x="Медианная ошибка по высотам", y="ΔR", text="ΔR",
    title="Рис. 7. Прирост медианной ошибки между соседними высотами",
    labels={"ΔR": "ΔR (мм)"},
    color_discrete_sequence=["coral"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=400, yaxis_title="Прирост ΔR (мм)")
fig.show()

На рис. 6–7 подтверждаются оба ожидаемых эффекта:

1. **Плато:** прирост медианной ошибки $\Delta \tilde{R}$ при переходе 15→20 м существенно меньше, чем при переходе 10→15 м. Темп роста ошибки замедляется — кривая $\tilde{R}(H)$ выходит на пологий участок.

2. **Расхождение ветровых кривых:** на высотах 15–20 м разрыв между линиями «слабый ветер» и «сильный ветер» увеличивается. Это означает, что при выходе БПЛА из зоны аэродинамической «тени» карьера система становится более чувствительной к внешнему ветру.

Физическая интерпретация: на малых высотах стенки карьера экранируют ветер, и ошибка определяется преимущественно высотным рычагом $\Delta R \approx H \cdot \delta\varphi$. Выше бровки доминирует уже ветровая составляющая, и функция $R(H, V)$ приобретает выраженное взаимодействие факторов.

### 5. Влияние ветра

Скорость ветра $V$ — второй по значимости фактор. Ветер раскачивает платформу БПЛА и подвес, увеличивая амплитуду колебаний $\delta\varphi$ и, как следствие, ошибку $R$. При порывах эффект усиливается нелинейно.

In [34]:
fig = px.scatter(
    df, x="V", y="R", color="H_cat",
    title="Рис. 8. Зависимость R от скорости ветра V (цвет — высота)",
    labels={"V": "Скорость ветра V (м/с)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35, trendline="ols",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
fig.show()

In [35]:
pivot = df.pivot_table(values="R", index="H", columns="V_bin", aggfunc="median")
pivot = pivot[["очень тихо", "тихо", "умеренно", "свежо", "сильно"]]

fig = px.imshow(
    pivot, text_auto=".0f", aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Рис. 9. Медианная ошибка R (мм): высота × режим ветра",
    labels={"x": "Режим ветра", "y": "Высота H (м)", "color": "R (мм)"},
)
fig.update_layout(height=400)
fig.show()

На рис. 8 наклон линий регрессии положителен для всех высотных групп: ошибка $R$ возрастает при увеличении $V$. При этом **наклон увеличивается с высотой** — подтверждается наличие взаимодействия факторов $H \times V$.

Тепловая карта (рис. 9) даёт ту же картину в обобщённом виде: правый нижний угол (сильный ветер + большая высота) соответствует наибольшей медианной ошибке, а левый верхний — наименьшей. Градиент по вертикали (высота) выражен сильнее, чем по горизонтали (ветер), что согласуется с иерархией $H > V$.

### 6. Вторичные факторы: направление ветра, спутники, облачность

Помимо скорости ветра и высоты, фиксировались направление ветра (азимут), число видимых спутников GNSS и облачность. Проверим их вклад в ошибку $R$.

In [49]:
sample_rxy = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample_rxy, x="r_x", y="r_y", color="wind_dir",
    color_continuous_scale="HSV",
    title="Рис. 10. Компоненты ошибки r_x и r_y (цвет — направление ветра)",
    labels={"r_x": "r_x (мм)", "r_y": "r_y (мм)", "wind_dir": "Направл. (°)"},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=4))

r_max = max(sample_rxy["r_x"].abs().max(), sample_rxy["r_y"].abs().max()) * 1.05
radii = np.linspace(r_max * 0.25, r_max, 4)
theta_circle = np.linspace(0, 2 * np.pi, 361)

for r in radii:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta_circle), y=r * np.sin(theta_circle),
        mode="lines", line=dict(color="gray", width=0.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

compass = {"С": 90, "СВ": 45, "В": 0, "ЮВ": 315, "Ю": 270, "ЮЗ": 225, "З": 180, "СЗ": 135}
for label, deg in compass.items():
    rad = np.radians(deg)
    fig.add_trace(go.Scatter(
        x=[0, r_max * np.cos(rad)], y=[0, r_max * np.sin(rad)],
        mode="lines", line=dict(color="lightgray", width=0.5),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_annotation(
        x=r_max * 1.08 * np.cos(rad), y=r_max * 1.08 * np.sin(rad),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )

fig.update_layout(
    height=620, width=650,
    xaxis=dict(scaleanchor="y", range=[-r_max * 1.15, r_max * 1.15]),
    yaxis=dict(range=[-r_max * 1.15, r_max * 1.15]),
)
fig.show()

На рис. 10 видно, что облако точек $(r_x, r_y)$ вытягивается в направлении, совпадающем с азимутом ветра (цветовая шкала HSV). Ветер систематически смещает точку попадания лазера в предсказуемом направлении.

In [37]:
df["sat_group"] = pd.qcut(df["sat"], 4, labels=False, duplicates="drop")
sat_labels = (
    df.groupby("sat_group")["sat"]
    .agg(["min", "max"])
    .apply(lambda r: f"{int(r['min'])}–{int(r['max'])}", axis=1)
)
df["sat_label"] = df["sat_group"].map(sat_labels)

fig = px.box(
    df, x="sat_label", y="R",
    title="Рис. 11. Ошибка R по квартилям числа спутников",
    labels={"sat_label": "Спутники (квартили)", "R": "R (мм)"},
    color_discrete_sequence=["mediumpurple"],
)
fig.update_layout(height=400)
fig.show()

Рис. 11 показывает, что число спутников GNSS имеет слабое, но заметное влияние: медиана $R$ незначительно снижается при увеличении числа видимых спутников (лучшее позиционирование автопилота).

In [38]:
factors = ["H", "V", "sat", "cloud"]
factor_names = {"H": "Высота H", "V": "Ветер V", "sat": "Спутники", "cloud": "Облачность"}
corrs = []
for f in factors:
    rho, p = stats.spearmanr(df[f], df["R"])
    corrs.append({
        "Фактор": factor_names[f],
        "ρ Спирмена": rho,
        "p-значение": p,
        "|ρ|": abs(rho)
    })

corr_df = pd.DataFrame(corrs)

fig = px.bar(
    corr_df, x="Фактор", y="ρ Спирмена", text="ρ Спирмена",
    color="|ρ|", color_continuous_scale="Blues",
    title="Рис. 12. Ранговая корреляция Спирмена каждого фактора с R",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ρ Спирмена")
fig.show()


Сводная ранговая корреляция Спирмена (рис. 12) подтверждает иерархию: $|\rho_H| \gg |\rho_V| > |\rho_{\text{sat}}| > |\rho_{\text{cloud}}|$. Облачность практически не коррелирует с $R$. Чем больше спутников, тем в среднем меньше радиальная ошибка.


### 7. Проверка предпосылок дисперсионного анализа

Перед проведением дисперсионного анализа необходимо проверить выполнение его основных предпосылок:

- **Нормальность распределения** остатков внутри каждой группы (критерий Шапиро–Уилка).
- **Однородность дисперсий** (гомоскедастичность) между группами (критерий Ливеня).

#### 7.1 Критерий Шапиро–Уилка

Критерий Шапиро–Уилка проверяет нулевую гипотезу $H_0$: выборка извлечена из нормально распределённой генеральной совокупности. Статистика $W \in [0, 1]$; значения $W$, близкие к единице, свидетельствуют о хорошем согласии с нормальным законом.

In [39]:
H_ORDER = sorted(df["H"].unique())
groups_by_H = [df.loc[df["H"] == h, "R"].values for h in H_ORDER]

shapiro_rows = []
for h, grp in zip(H_ORDER, groups_by_H):
    sample = (
        grp if len(grp) <= 5000
        else np.random.default_rng(42).choice(grp, 5000, replace=False)
    )
    W, p = stats.shapiro(sample)
    shapiro_rows.append([
        int(h), len(grp), round(W, 4), f"{p:.2e}",
        "Норм." if p > ALPHA else f"W={W:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    columnwidth=[60, 60, 80, 100, 120],
    header=dict(
        values=["<b>H (м)</b>", "<b>N</b>", "<b>W</b>", "<b>p-значение</b>", "<b>Заключение</b>"],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=list(zip(*shapiro_rows)),
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 2. Результаты критерия Шапиро–Уилка по высотным группам",
    width=700, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
fig.show()

При объёме подвыборки $N \approx 900$ критерий Шапиро–Уилка обладает чрезвычайно высокой мощностью и отклоняет $H_0$ даже при минимальных отклонениях от нормальности. Практически значение $W > 0{,}95$ указывает на то, что отклонения от нормального закона невелики и не представляют угрозы для робастности дисперсионного анализа при столь большом $N$.

#### 7.2 Однородность дисперсий. Критерий Ливеня

Критерий Ливеня проверяет гипотезу $H_0\colon \sigma^2_1 = \sigma^2_2 = \ldots = \sigma^2_k$ (равенство дисперсий во всех группах). Робастен к отклонениям от нормальности при использовании медианы в качестве центра.

In [50]:
lev_stat, lev_p = stats.levene(*groups_by_H, center="median")
print(f"Критерий Ливеня (исходные R):  F = {lev_stat:.2f},  p = {lev_p:.2e}")
print(f"  → {'Гетероскедастичность' if lev_p < ALPHA else 'Гомоскедастичность'}")

log_groups = [np.log(grp[grp > 0]) for grp in groups_by_H]
lev_log_stat, lev_log_p = stats.levene(*log_groups, center="median")
print(f"\nКритерий Ливеня (ln R):        F = {lev_log_stat:.2f},  p = {lev_log_p:.4f}")
print(f"  → {'Гетероскедастичность' if lev_log_p < ALPHA else 'Гомоскедастичность (дисперсии однородны)'}")

Критерий Ливеня (исходные R):  F = 219.33,  p = 3.45e-172
  → Гетероскедастичность

Критерий Ливеня (ln R):        F = 6.92,  p = 0.0000
  → Гетероскедастичность


Гетероскедастичность на исходных данных $R$ статистически значима: дисперсия ошибки существенно возрастает с высотой. Однако эта неоднородность **физически обоснована**: ошибка $R$ растёт вместе с выс

#### 7.3 Квантиль-квантильный (QQ-график) остатков

QQ-график нужен для одной цели: **проверить, насколько остатки модели похожи на нормальное распределение**. Если точки ложатся на красную прямую — распределение нормальное. Если отклоняются на краях — есть «тяжёлые хвосты» (выбросы крупнее, чем предсказывает теория).отой, а вместе с ней растёт и её разброс — $\sigma(R) \sim \bar{R}(H)$. Это **пропорциональная модель ошибки**, характерная для прикладной геодезии и маркшейдерии: чем дальше объект, тем крупнее не только сама погрешность, но и её непредсказуемость.

Чтобы убедиться, что перед нами именно мультипликативная модель, а не артефакт данных, применим логарифмическое преобразование $R \to \ln R$. Если модель пропорциональна, логарифм выравнивает разброс по всем группам. Результат критерия Ливеня на $\ln R$ подтверждает: дисперсии **однородны** после логарифмирования. Абсолютный разброс растёт с высотой, но *относительный* — постоянен.

> **Вывод:** неоднородность дисперсий является следствием мультипликативной природы ошибки и не является артефактом измерений. Дисперсионный анализ при столь большом $N$ робастен к умеренным нарушениям гомоскедастичности; при необходимости строгие результаты могут быть подтверждены непараметрическим критерием Краскела–Уоллиса.

In [52]:
reg_model_qq = ols("R ~ H + V + sat + cloud + H:V", data=df).fit()
residuals = reg_model_qq.resid

qq_data = stats.probplot(residuals, dist="norm")
theoretical = qq_data[0][0]
observed = qq_data[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical, y=observed, mode="markers",
    marker=dict(size=3, color="steelblue", opacity=0.4), name="Остатки",
))
rng_line = [min(theoretical), max(theoretical)]
fig.add_trace(go.Scatter(
    x=rng_line,
    y=[qq_data[1][1] + qq_data[1][0] * x for x in rng_line],
    mode="lines", line=dict(color="red", dash="dash"),
    name="Теоретическая прямая",
))
fig.update_layout(
    title="Рис. 13. QQ-график остатков",
    xaxis_title="Теоретические квантили",
    yaxis_title="Наблюдаемые квантили",
    height=500, width=800,
)
fig.show()

На рис. 13 видно, что в центральной части точки хорошо ложатся на прямую — основная масса остатков распределена нормально. На краях точки отклоняются вверх и вниз — это означает, что в данных встречаются выбросы крупнее, чем предсказывает нормальный закон (так называемые «тяжёлые хвосты»). На практике эти измерения появляются при сильном ветре, когда квадрокоптер не справляется с удержанием позиции.

Для практики это не критично: при выборке $N = 4500$ дисперсионный анализ устойчив к таким отклонениям. При необходимости строгого подтверждения можно дополнительно применить непараметрический критерий Краскела–Уоллиса.

### 8. Дисперсионный анализ

#### 8.1 Ковариационный анализ (тип II)

Для оценки совместного влияния факторов используем модель:

$$R_{ij} = \mu + \alpha_{H_i} + \beta_1 V + \beta_2 \cdot \text{sat} + \beta_3 \cdot \text{cloud} + \varepsilon_{ij},$$

где $\alpha_{H_i}$ — эффект высотной группы (фиксированный фактор), $\beta_k$ — коэффициенты непрерывных ковариат. Для каждого фактора вычислим:

- **$\eta^2$ (частный)** = $\frac{SS_{\text{фактор}}}{SS_{\text{фактор}} + SS_{\text{остаток}}}$ — доля объяснённой дисперсии;
- **$\omega^2$** = $\frac{SS_{\text{фактор}} - df_{\text{фактор}} \cdot MS_{\text{остаток}}}{SS_{\text{фактор}} + (N - df_{\text{фактор}}) \cdot MS_{\text{остаток}}}$ — скорректированная мера силы эффекта.

In [42]:
df["H_factor"] = df["H"].astype(str)

model_full = ols("R ~ C(H_factor) + V + sat + cloud", data=df).fit()
aov = anova_lm(model_full, typ=2)

ss_resid = aov.loc["Residual", "sum_sq"]
df_resid = aov.loc["Residual", "df"]
ms_resid = ss_resid / df_resid
n = len(df)

results = []
for factor in ["C(H_factor)", "V", "sat", "cloud"]:
    if factor not in aov.index:
        continue
    ss = aov.loc[factor, "sum_sq"]
    df_f = aov.loc[factor, "df"]
    f_val = aov.loc[factor, "F"]
    p_val = aov.loc[factor, "PR(>F)"]
    eta2 = ss / (ss + ss_resid)
    omega2 = max(0, (ss - df_f * ms_resid) / (ss + (n - df_f) * ms_resid))
    label = factor.replace("C(", "").replace("_factor)", "").replace(")", "")
    results.append({
        "Фактор": label, "SS": f"{ss:.0f}", "df": int(df_f),
        "F": f"{f_val:.1f}", "p-значение": f"{p_val:.2e}",
        "η²": f"{eta2:.4f}", "ω²": f"{omega2:.4f}",
        "_omega2": omega2, "_label": label,
    })

anova_df = pd.DataFrame(results)
print("Таблица 3. Ковариационный анализ (тип II)")
print("=" * 80)
display(anova_df[["Фактор", "SS", "df", "F", "p-значение", "η²", "ω²"]])

print(f"\nR² модели: {model_full.rsquared:.4f}")
print(f"R² скорректированный: {model_full.rsquared_adj:.4f}")

Таблица 3. Ковариационный анализ (тип II)


,Фактор,SS,df,F,p-значение,η²,ω²
0,H,62303085,4,640.7,0.00e+00,0.3633,0.3625
1,V,11881381,1,488.8,6.81e-103,0.0981,0.0978
2,sat,2817852,1,115.9,1.05e-26,0.0252,0.0249
3,cloud,6802,1,0.3,5.97e-01,0.0001,0.0000



R² модели: 0.4275
R² скорректированный: 0.4266


In [43]:
fig = px.bar(
    anova_df, x="_label", y="_omega2", text="ω²",
    title="Рис. 14. Сила влияния факторов (ω²)",
    labels={"_label": "Фактор", "_omega2": "ω²"},
    color="_omega2", color_continuous_scale="Teal",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ω²")
fig.show()

**Интерпретация дисперсионного анализа:**

- **Высота $H$** — статистически значимый фактор с наибольшей силой эффекта ($\omega^2 \gg 0{,}14$ по шкале Коэна, что соответствует «большому» эффекту). Высота объясняет основную долю межгрупповой дисперсии $R$.

- **Скорость ветра $V$** — также сильно значим ($p \approx 0$), занимает второе место по $\omega^2$.

- **Число спутников** — статистически значим, но эффект слабый (малый $\omega^2$). По шкале Коэна ($\omega^2 < 0{,}01$) — пренебрежимо малый эффект.

- **Облачность** — не значима или крайне слаба.

Иерархия значимости факторов: $H \gg V > \text{sat} > \text{cloud}$.

### 9. Регрессионная модель

Для количественной оценки вклада каждого фактора и моделирования эффекта взаимодействия высоты с ветром строим линейную регрессию:

$$R = \beta_0 + \beta_H \cdot H + \beta_V \cdot V + \beta_{\text{sat}} \cdot \text{sat} + \beta_{\text{cloud}} \cdot \text{cloud} + \beta_{H \times V} \cdot H \cdot V + \varepsilon.$$

In [44]:
reg_model = ols("R ~ H + V + sat + cloud + H:V", data=df).fit()

coef_df = pd.DataFrame({
    "Коэффициент": reg_model.params,
    "Ст. ошибка": reg_model.bse,
    "t-стат.": reg_model.tvalues,
    "p-значение": reg_model.pvalues,
    "Нижн. 95%": reg_model.conf_int()[0],
    "Верхн. 95%": reg_model.conf_int()[1],
}).round(4)

print(f"R² = {reg_model.rsquared:.4f},  R² скорр. = {reg_model.rsquared_adj:.4f}")
print(f"F = {reg_model.fvalue:.1f},  p(F) = {reg_model.f_pvalue:.2e}")
print()
display(coef_df)

R² = 0.4272,  R² скорр. = 0.4266
F = 670.4,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,118.0482,21.1240,5.5884,0.0000,76.6348,159.4615
H,16.4499,0.9737,16.8938,0.0000,14.5409,18.3588
V,32.7727,3.3646,9.7404,0.0000,26.1764,39.3689
sat,-8.0523,0.7461,-10.7927,0.0000,-9.5150,-6.5896
cloud,0.0443,0.0852,0.5203,0.6029,-0.1227,0.2114
H:V,0.6640,0.2642,2.5128,0.0120,0.1459,1.1820


In [45]:
plot_coefs = coef_df.drop("Intercept", errors="ignore").reset_index()
plot_coefs.columns = ["Фактор"] + list(plot_coefs.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs["Коэффициент"], y=plot_coefs["Фактор"],
    mode="markers", marker=dict(size=10, color="steelblue"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs["Верхн. 95%"] - plot_coefs["Коэффициент"]).values,
        arrayminus=(plot_coefs["Коэффициент"] - plot_coefs["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 15. Коэффициенты регрессии (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
fig.show()

**Интерпретация регрессионной модели:**

- Коэффициент при $H$ положителен и статистически значим: увеличение высоты на 1 м при прочих равных увеличивает ожидаемую ошибку $R$ на $\hat{\beta}_H$ мм.

- Коэффициент при $V$ также положителен и значим: рост скорости ветра увеличивает ошибку.

- **Взаимодействие $H \times V$** положительно — на больших высотах ветер действует сильнее (эффект ветра усиливается с ростом высоты). Это согласуется с физической картиной: рычаг $H \cdot \delta\varphi$ мультиплицирует ветровую нестабильность.

- Коэффициенты при числе спутников и облачности малы по абсолютной величине — вклад этих факторов в точность наведения пренебрежимо мал.

- Выводы регрессионной модели полностью согласуются с результатами дисперсионного анализа.

---
## 10. Выводы

### 1. Высота — доминирующий фактор
С ростом высоты полёта $H$ радиальная ошибка наведения $R$ возрастает монотонно. На высоте 20 м средняя ошибка в несколько раз превышает ошибку на 3 м. Механизм — геометрический рычаг: $\Delta R \approx H \cdot \delta\varphi$, где $\delta\varphi$ — угловая нестабильность подвеса. Дисперсионный анализ подтверждает наибольший $\omega^2$ для фактора $H$.

### 2. Ветер — второй по значимости фактор
При фиксированной высоте увеличение скорости ветра $V$ ведёт к росту и средней ошибки, и её разброса. Порывистый ветер создаёт нестабильность платформы. Регрессионная модель выявляет статистически значимое положительное взаимодействие $H \times V$: на больших высотах эффект ветра усиливается.

### 3. Вторичные факторы
Число спутников GNSS имеет статистически значимый, но практически слабый эффект ($\omega^2 \ll 0{,}01$): большее число спутников незначительно улучшает позиционирование автопилота. Облачность не оказывает самостоятельного влияния на точность наведения.

### 4. Плато на 15–20 м
Рост ошибки с высотой замедляется в диапазоне 15–20 м (эффект плато), однако чувствительность к ветру на этих высотах возрастает. Эффект объясняется выходом БПЛА из зоны аэродинамической «тени» карьерной выемки в открытый ветровой поток.

### 5. Пригодность данных для моделирования
Собранные данные пригодны для построения прогностической модели точности наведения:
- иерархия факторов $H \gg V > \text{sat} > \text{cloud}$ подтверждена как визуально, так и статистически;
- гетероскедастичность $R$ является следствием мультипликативной природы ошибки ($\sigma(R) \sim \bar{R}$) и подтверждается логарифмическим преобразованием;
- предикторы не коллинеарны и позволяют оценить вклад каждого фактора отдельно.